In [2]:
import os
import shutil
import pandas as pd

def format_val_directory(val_dir):
    val_annotations = os.path.join(val_dir, 'val_annotations.txt')
    val_images_dir = os.path.join(val_dir, 'images')

    # 如果 annotations 文件不存在，说明已经整理过了
    if not os.path.exists(val_annotations):
        print("验证集已经整理过啦！")
        return

    # 读取标签映射文件
    val_df = pd.read_csv(val_annotations, sep='\t', header=None,
                         names=['File', 'Class', 'X', 'Y', 'H', 'W'])

    print("开始整理验证集目录...")
    for index, row in val_df.iterrows():
        file_name = row['File']
        class_name = row['Class']

        # 创建类别子文件夹
        class_dir = os.path.join(val_dir, class_name)
        os.makedirs(class_dir, exist_ok=True)

        # 将图片从 images/ 移动到对应的 class_dir/
        src = os.path.join(val_images_dir, file_name)
        dst = os.path.join(class_dir, file_name)
        if os.path.exists(src):
            shutil.move(src, dst)

    # 删除没用的空文件夹和 txt 文件
    os.rmdir(val_images_dir)
    os.remove(val_annotations)
    print("验证集整理完成！现在的结构与 train 完全一致。")

# 执行整理
format_val_directory('./Tiny_Images/tiny-imagenet-200/val')

开始整理验证集目录...
验证集整理完成！现在的结构与 train 完全一致。


In [4]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. 定义数据增强与预处理 (Transform)
# Tiny ImageNet 图片很小(64x64)，不要再做额外的 Resize 缩小它
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), # 训练时加入随机水平翻转防过拟合
    transforms.ToTensor(),             # [0, 255] 像素值转为 [0.0, 1.0] 的张量
    # 标准化：ImageNet 通用的均值和标准差
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 验证集只需要 ToTensor 和 Normalize，绝对不能做随机数据增强
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 2. 实例化 Dataset
data_root = './tiny-imagenet-200'
train_dataset = datasets.ImageFolder(root=f'{data_root}/train', transform=train_transform)
val_dataset = datasets.ImageFolder(root=f'{data_root}/val', transform=val_transform)

# 3. 封装进 DataLoader
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"训练集类别数: {len(train_dataset.classes)}, 图片总数: {len(train_dataset)}")
print(f"验证集图片总数: {len(val_dataset)}")

# 4. 测试抓取一个 Batch
images, labels = next(iter(train_loader))
print(f"Image Batch Shape: {images.shape}") # 预期输出: [64, 3, 64, 64]
print(f"Label Batch Shape: {labels.shape}") # 预期输出: [64]


FileNotFoundError: [Errno 2] No such file or directory: './tiny-imagenet-200/train'